# Topic: JSON Serialization - json dumps/loads, stream dump/load, and custom JSONEncoder subclasses
 
## 1. DEFINITION & BUILT-IN DATA TYPES
- **JSON (JavaScript Object Notation)**: A lightweight data-interchange format.
- **json module**: Python's standard library module for converting between Python data types and JSON strings/streams:
  - **`json.dumps(obj)`**: Serializes a Python object to a JSON formatted **string**.
  - **`json.dump(obj, fp)`**: Serializes a Python object directly to a open **file stream** (writing).
  - **`json.loads(json_string)`**: Parses a JSON formatted string back into Python objects.
  - **`json.load(fp)`**: Parses a JSON formatted file stream directly.
- **Type Mapping**:
  - Dict -> object, List/Tuple -> array, Str -> string, Int/Float -> number, True/False -> true/false, None -> null.
 
## 2. CUSTOM OBJECTS SERIALIZATION (JSONEncoder)
- **The Problem**: If you pass a custom class instance to `json.dumps()`, Python raises a `TypeError: Object of type X is not JSON serializable`.
- **The Solution**: Subclass **`json.JSONEncoder`** and override the **`default(self, o)`** method to return a serializable primitive type (like a dictionary of object attributes). Pass this encoder class to the dumps call using `cls=CustomEncoder`.
 
---


In [ ]:
import json
import os

json_file = "data.json"

# 1. Standard dumps and loads
data_dict = {
    "name": "Alice",
    "is_active": True,
    "tags": ["python", "json"],
    "score": 98.5,
    "credentials": None
}

print("--- Standard JSON String Serialization ---")
json_str = json.dumps(data_dict, indent=2)
print("Serialized String:")
print(json_str)

parsed_dict = json.loads(json_str)
print(f"\nParsed dict matches original? {parsed_dict == data_dict}")



In [ ]:
# 2. Writing and Reading JSON Streams (dump/load)
print("\n--- File Stream JSON Serialization (dump/load) ---")
with open(json_file, "w", encoding="utf-8") as f:
    json.dump(data_dict, f)  # Writes compressed single-line JSON

# Read from stream
with open(json_file, "r", encoding="utf-8") as f:
    loaded_data = json.load(f)
    print(f"Loaded JSON from stream: {loaded_data}")



In [ ]:
# 3. Custom Object Serialization using JSONEncoder
class Student:
    """Custom class that is not serializable by default."""
    def __init__(self, name, age):
        self.name = name
        self.age = age

class StudentEncoder(json.JSONEncoder):
    """Custom JSONEncoder subclass."""
    def default(self, obj):
        if isinstance(obj, Student):
            # Return a serializable representation (like a dictionary of attributes)
            return {"__class__": "Student", "name": obj.name, "age": obj.age}
        # Fall back to parent default for standard types
        return super().default(obj)

print("\n--- Custom Class JSON Serialization ---")
alice = Student("Alice", 21)

try:
    # This will fail with TypeError
    json.dumps(alice)
except TypeError as e:
    print(f"Caught expected TypeError: {e}")

# This succeeds using custom StudentEncoder
serialized_student = json.dumps(alice, cls=StudentEncoder, indent=2)
print("Serialized Custom Student:")
print(serialized_student)



In [ ]:
# 4. Clean up
if os.path.exists(json_file):
    os.remove(json_file)
